# Notebook 4: Sharding and Parallelism

Unlike PyTorch DDP/FSDP which wraps models in heavy class logic, JAX separates the numerical program from its physical data layout. Simply utilizes `jax.sharding` and `PartitionSpec` to dictate how matrices distribute across TPUs/GPUs.



In [4]:
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), '.', 'third-party', 'simply'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import jax
import jax.numpy as jnp
import numpy as np
from simply.utils import sharding as sharding_lib
print(f"Devices Available: {jax.device_count()}")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Devices Available: 1


## 1. JAX Mesh Configurations
A `jax.sharding.Mesh` is a logical multi-dimensional grid of physical devices. Simply models typically specify "data", "fsdp", and "tensor" parallel mesh axes.

### Exercise 1: Build a Mesh Layout
Use `jax.devices()` to retrieve an array of current hardware accelerators. Restructure it into a 2D mesh layout mapped via `numpy.reshape` and bind it to a `jax.sharding.Mesh` using axis names ('fsdp', 'tp').

In [7]:
# Exercise 1 Code

from jax.sharding import Mesh, PartitionSpec, NamedSharding
import numpy as np

devices = jax.devices()
num_devices = len(devices)
print(f"Available devices for mesh: {num_devices}")

# TODO: Using `np.array(devices).reshape(...)`, create a logical grid if possible.
# Since we might only have 1 device locally, reshape logic needs to be robust.
grid = np.array(devices).reshape(num_devices, 1)

# Then instantiate `mesh = Mesh(grid, axis_names=('fsdp', 'tp'))`
mesh = Mesh(grid, axis_names=('fsdp', 'tp'))

if mesh is not None:
    print(f"Created Mesh:\n{mesh} for grid {grid}")

Available devices for mesh: 1
Created Mesh:
Mesh('fsdp': 1, 'tp': 1, axis_types=(Auto, Auto)) for grid [[CpuDevice(id=0)]]


## 2. Partition Specs
Given a tensor, `PartitionSpec` defines which logical mesh axes divide which array dimensions. Simply's config files are full of `PartitionSpec('fsdp', None)` strings.

### Exercise 2: Defining Layout Variables
Write a PartitionSpec that shards the first axis of a Batch-Sequence-Embedding `[B, S, E]` array over the `fsdp` mesh axis, and leaves the sequence and embedding axes entirely replicated (`None`).

In [8]:
# Exercise 2 Code

# PartitionSpec maps index -> mesh axis.
# TODO: Create `spec` such that dimension 0 is mapped to 'fsdp',
# dimension 1 is replicated (None), and dimension 2 is replicated (None).
spec = PartitionSpec('fsdp', None, None)

if spec is not None:
    print(f"Partition Specification: {spec}")

Partition Specification: P('fsdp', None, None)


## 3. Sharding Constraints
Simply uses `jax.lax.with_sharding_constraint` to explicitly force the JAX compiler to adopt a specific memory layout. It's often found inside `.apply()` before intense matrix operations.

### Exercise 3: Apply Sharding to a Local Array
Given an array $X$, construct a `NamedSharding` object from the Mesh and PartitionSpec created above, then constrain $X$ utilizing `with_sharding_constraint`.

In [9]:
# Exercise 3 Code

from jax import lax

X = jnp.ones((8, 128, 512))

# We will use the `mesh` and `spec` from previous steps if available.
constrained_x = None

if mesh is not None and spec is not None:
    # TODO: Create NamedSharding layout 
    my_sharding = NamedSharding(mesh, spec)
    
    # TODO: Apply the constraint via `lax.with_sharding_constraint(X, my_sharding)`
    constrained_x = lax.with_sharding_constraint(X, my_sharding)

if constrained_x is not None:
    print("Successfully applied Sharding Constraint.")
    print("Target Array Layout:", constrained_x.sharding)

Successfully applied Sharding Constraint.
Target Array Layout: NamedSharding(mesh=Mesh('fsdp': 1, 'tp': 1, axis_types=(Auto, Auto)), spec=P('fsdp', None, None), memory_kind=device)


## 4. Annotated Arrays & Sharding Translation
Look closely at `simply/utils/sharding.py`. Simply bridges purely functional arrays and PyTree metadata efficiently via `AnnotatedArray`s. During state loading, string annotations like `'.io'` map to physical properties.

### Exercise 4: Parse an Annotation
Construct an `AnnotatedArray` and explore the `simply.utils.sharding.get_partition_axis` utility to translate symbolic rules automatically into `PartitionSpec` tuples.

In [15]:
?sharding_lib.get_partition_axis

Signature:
sharding_lib.get_partition_axis(
    partition: None | collections.abc.Sequence[None | str | collections.abc.Sequence[str]],
    axis: int,
) -> str | collections.abc.Sequence[str] | None
Docstring: <no docstring>
File:      /mnt/d/Code/jax-code/third-party/simply/simply/utils/sharding.py
Type:      function

In [19]:
from simply.utils import common
from simply.utils import sharding as sharding_lib
from simply.config_lib import gspmd_sharding
# 1. Use AttributeDict instead of SimplyConfig to avoid the Any error
config = common.AttributeDict()
config.fsdp_axis = 'f'
config.tp_axis = 't'
# 2. Create the AnnotatedArray
weight_data = jnp.zeros((16, 16))
weight = common.AnnotatedArray.create(weight_data, dim_annotation='io')
# 3. Translate the symbolic annotation. 
# In Simply, we often use sharding_lib.partition_spec to convert 
# simple annotations into PartitionSpecs. 
# Here we can look up the partition for the 'io' annotation from a real sharding config.
translation = sharding_lib.get_partition_axis('io', axis=1) # Which is ('data', 'model')
if translation is not None:
    print(f"Annotation 'io' maps to partition spec: {translation}")


Annotation 'io' maps to partition spec: o


In [10]:
# Exercise 4 Code

from simply.utils import common
from simply.utils import sharding as sharding_lib
from simply.config_lib import SimplyConfig

weight = common.AnnotatedArray.create(jnp.zeros((16, 16)), dim_annotation='io')
config = SimplyConfig()
config.fsdp_axis = 'f'
config.tp_axis = 't'

# Note: This simply translates a symbolic config into actionable JAX specifications.
# Call `sharding_lib.get_partition_axis('io', sharding_config=config)` 
translation = None

if translation is not None:
    print("Translated '.io' annotation maps to partition spec attributes:", translation)

TypeError: Any cannot be instantiated

## 5. SPMD Execution and Shard Map
Finally, the actual operations occur across all shards in SPMD paradigm (Single Program Multiple Data). JAX executes your logic globally, but sometimes you need fine-grained local control. Simply provides patterns using `shard_map`.

### Exercise 5: Simulated SPMD Execution
Write a simple kernel and wrap it in `jax.experimental.shard_map.shard_map` (simulated). Validate that within the map, the shape of the array represents a *local* shard, not the global batch.

In [ ]:
# Exercise 5 Code

from jax.experimental.shard_map import shard_map

def my_kernel(local_x):
    # This function operates on a shard
    return local_x * 2.0

global_array = jnp.ones((8, 128))

# If a mesh exists, one could apply `shard_map(my_kernel, mesh, in_specs=..., out_specs=...)`
# TODO: Assuming the Mesh is set up, inspect the global vs local shapes logically.
mapped_kernel_fn = None

if mesh is not None and spec is not None:
    # Define mapping constraints:
    mapped_kernel_fn = shard_map(
        my_kernel, 
        mesh=mesh,
        in_specs=PartitionSpec('fsdp', None),
        out_specs=PartitionSpec('fsdp', None)
    )
    
    # It's an executable function! Run it on global_array:
    # res = mapped_kernel_fn(global_array)
    pass

if mapped_kernel_fn is not None:
    print("Successfully instantiated shard_map JIT execution!")